In [ ]:
# # Core dependencies
# pip install torch torchvision gymnasium numpy matplotlib pillow imageio pygame
# pip install torch torchvision
# pip install gymnasium
# pip install pygame-learning-environment
# pip install numpy matplotlib

# # For visualization and analysis
# pip install pillow
# pip install imageio  # for gif creation

In [19]:
import torch
import gymnasium as gym
import pygame
from ple import PLE
from ple.games.pixelcopter import Pixelcopter
import numpy as np

# Load the trained model
# Note: Adjust path based on your model file structure
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# model = torch.load("pixelcopter_reinforce_model.pth", map_location=device)
# model.eval()


# Create the environment
def create_pixelcopter_env():
    game = Pixelcopter()
    env = PLE(game, fps=30)  # Set display=False for headless
    return env


# Run trained agent
def run_agent(model, env, episodes=5):
    total_rewards = []

    for episode in range(episodes):
        env.reset_game()
        episode_reward = 0

        while not env.game_over():
            # Get current state
            state = env.getScreenRGB()  # or env.getGameState() if using features
            state = preprocess_state(state)  # Apply your preprocessing

            # Convert to tensor
            state_tensor = torch.FloatTensor(state).unsqueeze(0).to(device)

            # Get action probabilities
            with torch.no_grad():
                action_probs = model(state_tensor)
                action = torch.multinomial(action_probs, 1).item()

            # Execute action (0: do nothing, 1: thrust)
            reward = env.act(action)
            episode_reward += reward

        total_rewards.append(episode_reward)
        print(f"Episode {episode + 1}: Reward = {episode_reward:.2f}")

    mean_reward = np.mean(total_rewards)
    std_reward = np.std(total_rewards)
    print(f"\nAverage Performance: {mean_reward:.2f} ± {std_reward:.2f}")

    return total_rewards


# Preprocessing function (adjust based on your model's input requirements)
def preprocess_state(state):
    """
    Preprocess the game state for the neural network
    This should match the preprocessing used during training
    """
    if isinstance(state, np.ndarray) and len(state.shape) == 3:
        # If using image input
        state = np.transpose(state, (2, 0, 1))  # Channel first
        state = state / 255.0  # Normalize pixels
        return state.flatten()  # or keep as image depending on model
    else:
        # If using game state features
        return np.array(list(state.values()))



In [14]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from collections import deque


class PolicyNetwork(nn.Module):
    def __init__(self, state_size, action_size, hidden_size=64):
        super(PolicyNetwork, self).__init__()
        self.fc1 = nn.Linear(state_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.fc3 = nn.Linear(hidden_size, action_size)
        self.softmax = nn.Softmax(dim=1)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = self.fc3(x)
        return self.softmax(x)


class REINFORCEAgent:
    def __init__(self, state_size, action_size, lr=0.001):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.policy_net = PolicyNetwork(state_size, action_size).to(self.device)
        self.optimizer = optim.Adam(self.policy_net.parameters(), lr=lr)

        self.saved_log_probs = []
        self.rewards = []

    def select_action(self, state):
        state = torch.FloatTensor(state).unsqueeze(0).to(self.device)
        probs = self.policy_net(state)
        action = torch.multinomial(probs, 1)

        self.saved_log_probs.append(torch.log(probs.squeeze(0)[action]))
        return action.item()

    def update_policy(self, gamma=0.99):
        # Calculate discounted rewards
        discounted_rewards = []
        R = 0

        for r in reversed(self.rewards):
            R = r + gamma * R
            discounted_rewards.insert(0, R)

        # Normalize rewards
        discounted_rewards = torch.FloatTensor(discounted_rewards).to(self.device)
        discounted_rewards = (discounted_rewards - discounted_rewards.mean()) / (
            discounted_rewards.std() + 1e-8
        )

        # Calculate policy loss
        policy_loss = []
        for log_prob, reward in zip(self.saved_log_probs, discounted_rewards):
            policy_loss.append(-log_prob * reward)

        # Update policy
        self.optimizer.zero_grad()
        policy_loss = torch.cat(policy_loss).sum()
        policy_loss.backward()
        self.optimizer.step()

        # Clear episode data
        self.saved_log_probs.clear()
        self.rewards.clear()

        return policy_loss.item()


In [ ]:
def train_agent(episodes=2000, log_every=100):
    env = create_pixelcopter_env()
    env.init()
    acts = env.getActionSet()
    print("Action Set:", acts)

    # Determine state size based on your preprocessing
    state = env.getGameState()  # or env.getScreenRGB() if using images
    # state = env.getScreenRGB()  # or env.getScreenRGB() if using images
    state_size = len(preprocess_state(state))  # Adjust as needed
    action_size = 2  # do nothing, thrust

    agent = REINFORCEAgent(state_size, action_size)

    scores = deque(maxlen=2000)

    for episode in range(episodes):
        env.reset_game()
        episode_reward = 0

        while not env.game_over():
            state = preprocess_state(env.getGameState())  # or env.getScreenRGB() if using images
            # state = preprocess_state(env.getScreenRGB())  # or env.getScreenRGB() if using images
            action = agent.select_action(state)
            # print(f"Episode {episode}, Action taken: {action}")
            reward = env.act(acts[action])
            agent.rewards.append(reward)
            episode_reward += reward

        # Update policy after each episode
        loss = agent.update_policy()
        scores.append(episode_reward)

        if episode % log_every == 0:
            avg_score = np.mean(scores)
            print(
                f"Episode {episode}, Average Score: {avg_score:.2f}, Loss: {loss:.4f}"
            )

    # Save the trained model
    torch.save(agent.policy_net, "pixelcopter_reinforce_model.pth")
    return agent


# Train a new agent
# trained_agent = train_agent()


In [18]:
trained_agent = train_agent(episodes=50000, log_every=500)


Action Set: [119, None]
Episode 0, Average Score: -5.00, Loss: 0.0258
Episode 500, Average Score: -2.65, Loss: 0.0000
Episode 1000, Average Score: -2.65, Loss: 0.0000
Episode 1500, Average Score: -2.65, Loss: 0.0000
Episode 2000, Average Score: -2.64, Loss: 0.0000
Episode 2500, Average Score: -2.65, Loss: 0.0000
Episode 3000, Average Score: -2.66, Loss: 0.0000
Episode 3500, Average Score: -2.65, Loss: 0.0000
Episode 4000, Average Score: -2.66, Loss: 0.0000
Episode 4500, Average Score: -2.65, Loss: 0.0000
Episode 5000, Average Score: -2.65, Loss: 0.0000
Episode 5500, Average Score: -2.64, Loss: 0.0000
Episode 6000, Average Score: -2.64, Loss: 0.0000
Episode 6500, Average Score: -2.65, Loss: 0.0000


KeyboardInterrupt: 

In [8]:
# Initialize environment
env = create_pixelcopter_env()
env.init()

# Run the agent
rewards = run_agent(trained_agent.policy_net, env, episodes=10)


Episode 1: Reward = -3.00
Episode 2: Reward = -2.00
Episode 3: Reward = -3.00
Episode 4: Reward = -3.00
Episode 5: Reward = -2.00
Episode 6: Reward = -3.00
Episode 7: Reward = -3.00
Episode 8: Reward = -3.00
Episode 9: Reward = -2.00
Episode 10: Reward = -3.00

Average Performance: -2.70 ± 0.46


In [ ]:
import matplotlib.pyplot as plt


def evaluate_agent_detailed(model, env, episodes=50):
    """Detailed evaluation with statistics and visualization"""
    rewards = []
    episode_lengths = []

    for episode in range(episodes):
        env.reset_game()
        episode_reward = 0
        steps = 0

        while not env.game_over():
            state = preprocess_state(env.getScreenRGB())
            state_tensor = torch.FloatTensor(state).unsqueeze(0)

            with torch.no_grad():
                action_probs = model(state_tensor)
                action = torch.multinomial(action_probs, 1).item()

            reward = env.act(action)
            episode_reward += reward
            steps += 1

        rewards.append(episode_reward)
        episode_lengths.append(steps)

        if (episode + 1) % 10 == 0:
            print(f"Episodes {episode + 1}/{episodes} completed")

    # Statistical analysis
    mean_reward = np.mean(rewards)
    std_reward = np.std(rewards)
    median_reward = np.median(rewards)
    max_reward = np.max(rewards)
    min_reward = np.min(rewards)

    mean_length = np.mean(episode_lengths)

    print(f"\n--- Evaluation Results ---")
    print(f"Episodes: {episodes}")
    print(f"Mean Reward: {mean_reward:.2f} ± {std_reward:.2f}")
    print(f"Median Reward: {median_reward:.2f}")
    print(f"Max Reward: {max_reward:.2f}")
    print(f"Min Reward: {min_reward:.2f}")
    print(f"Mean Episode Length: {mean_length:.1f} steps")

    # Visualization
    plt.figure(figsize=(12, 4))

    plt.subplot(1, 2, 1)
    plt.plot(rewards)
    plt.axhline(
        y=mean_reward, color="r", linestyle="--", label=f"Mean: {mean_reward:.2f}"
    )
    plt.title("Episode Rewards")
    plt.xlabel("Episode")
    plt.ylabel("Reward")
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.hist(rewards, bins=20, alpha=0.7)
    plt.axvline(
        x=mean_reward, color="r", linestyle="--", label=f"Mean: {mean_reward:.2f}"
    )
    plt.title("Reward Distribution")
    plt.xlabel("Reward")
    plt.ylabel("Frequency")
    plt.legend()

    plt.tight_layout()
    plt.show()

    return {
        "rewards": rewards,
        "episode_lengths": episode_lengths,
        "stats": {
            "mean": mean_reward,
            "std": std_reward,
            "median": median_reward,
            "max": max_reward,
            "min": min_reward,
        },
    }


# Run detailed evaluation
# results = evaluate_agent_detailed(model, env, episodes=100)
